## Streszczenie

Celem projektu jest wytrenowanie dwóch modeli, generujących muzykę, korzytsając z bibliotek torch. Do trenowania modeli LSTM oraz transformerowego użyta została oprawa muzyczna gier DOOM oraz DOOM 2 zapisanych w formacie mid. Odrzucono powtarzające się i odstające od reszty utwory a następnie utworzeno na ich podstawie słownik tokenów. Trening odbywał sie w wielu wariantach, aby umożliwić modelom jak najlepsze odwzorowanie klasycznej oprawy muzycznej gier z serii DOOM.

## 2. Wstęp / wprowadzenie

Generowanie muzyki z użyciem sieci neuronowych stanowi wymagające zadanie z zakresu modelowania sekwencji, ponieważ model musi uchwycić zarówno lokalne zależności między kolejnymi zdarzeniami, jak i dłuższą strukturę rytmiczną oraz motywiczną utworu. W przypadku muzyki symbolicznej problem ten rozpatruje się najczęściej na poziomie zdarzeń MIDI, co pozwala operować na reprezentacji nut, instrumentów i przesunięć czasowych zamiast na surowym sygnale audio.

W literaturze stosuje się w tym celu różne architektury autoregresyjne, w szczególności modele rekurencyjne typu LSTM oraz nowsze modele oparte na mechanizmie self-attention, takie jak Transformer. Pierwsze z nich dobrze modelują lokalną ciągłość sekwencji, natomiast drugie lepiej radzą sobie z uchwyceniem zależności długoterminowych i bardziej złożonej struktury muzycznej.

W projekcie rozważono oba podejścia w zadaniu generowania muzyki w stylistyce ścieżki dźwiękowej gier *Doom* i *Doom II* autorstwa Bobby'ego Prince'a. Wybrany zbiór danych jest stosunkowo mały, ale jednocześnie spójny stylistycznie i dostępny w formacie MIDI, co czyni go dobrym materiałem do eksperymentów z modelowaniem muzyki symbolicznej.

## 3. Opis kontekstu i celu

Projekt wpisuje się w obszar generowania muzyki symbolicznej, w którym utwór reprezentowany jest jako sekwencja zdarzeń MIDI przewidywanych na podstawie wcześniejszego kontekstu. Jest to zadanie zbliżone formalnie do modelowania języka, jednak w praktyce wymaga uchwycenia nie tylko kolejności zdarzeń, ale również regularności rytmicznej, powtarzalności motywów i spójności w dłuższych odcinkach sekwencji.

Istotnym kontekstem projektu jest porównanie dwóch klas architektur używanych w generowaniu muzyki: modeli rekurencyjnych LSTM oraz modeli opartych na mechanizmie uwagi, takich jak Transformer. LSTM stanowi klasyczne podejście do modelowania sekwencji i dobrze nadaje się do uchwycenia zależności lokalnych, natomiast Transformer lepiej radzi sobie z modelowaniem relacji długoterminowych dzięki bezpośredniemu dostępowi do całego kontekstu wejściowego.

Jako materiał treningowy wybrano utwory z gier *Doom* i *Doom II* autorstwa Bobby'ego Prince’a. Zbiór ten jest relatywnie mały, lecz jednorodny stylistycznie i dostępny w formacie MIDI.

Głównym celem projektu było zbudowanie i przetestowanie modeli zdolnych do generowania nowych sekwencji MIDI utrzymanych w stylistyce oryginalnej muzyki z serii *Doom*. Celem dodatkowym było sprawdzenie, jak model LSTM i model Transformer zachowują się w warunkach małego zbioru danych oraz które cechy generowanej muzyki każdy z nich odwzorowuje lepiej.

Od strony praktycznej projekt obejmował cały proces eksperymentalny: przygotowanie danych, tokenizację zdarzeń, implementację i trening modeli, a następnie generowanie i odsłuch otrzymanych sekwencji. Ocena jakości miała charakter zarówno ilościowy, na podstawie funkcji straty, jak i jakościowy, przez analizę odsłuchową wygenerowanej muzyki.

## 5. Opis danych

W projekcie wykorzystano zbiór plików MIDI zawierających muzykę z gier *Doom* i *Doom II*. Dane zostały pobrane z dwóch archiwów: [`doommusic.zip`](https://archive.org/download/doommusic) oraz [`doom2music.zip`](https://archive.org/download/doom2music), a po ich rozpakowaniu uzyskano łącznie 67 surowych plików MIDI.

Wybrany materiał ma charakter symboliczny, co oznacza, że każdy utwór zapisany jest jako zbiór zdarzeń muzycznych, takich jak nuty, instrumenty czy informacje czasowe, a nie jako sygnał audio. Taka forma danych jest wygodna w zadaniu generowania muzyki, ponieważ umożliwia analizę struktury utworu bez konieczności przetwarzania surowej fali dźwiękowej.

W ramach wstępnej analizy dla każdego pliku wyznaczono podstawowe statystyki, obejmujące między innymi czas trwania, liczbę instrumentów, liczbę nut, średnie tempo oraz informację o obecności ścieżki perkusyjnej. Na tej podstawie przeprowadzono czyszczenie zbioru: wykonano deduplikację po liczbie nut oraz odrzucono utwory krótsze niż 60 sekund. Usunięto też plik `dbunny.mid`, który nie był spójny stylistycznie z resztą danych. W rezultacie końcowy zbiór zawierał 42 utwory przeznaczone do dalszego przetwarzania.

Po przygotowaniu danych każdy utwór został zapisany jako sekwencja tokenów reprezentujących zdarzenia muzyczne. Łącznie otrzymano 382 035 tokenów dla 42 utworów, co wskazuje, że mimo niewielkiej liczby plików zbiór zawiera stosunkowo dużą liczbę elementów sekwencyjnych do uczenia modeli.

Dodatkowo wykonano prostą analizę eksploracyjną zbioru danych w celu lepszego zrozumienia jego struktury. Rozkład liczby nut w utworach pozwala ocenić zróżnicowanie gęstości muzycznej między ścieżkami, natomiast rozkład długości utworów pokazuje, że zbiór zawiera zarówno krótsze, jak i wyraźnie dłuższe kompozycje. Uzupełnieniem tej analizy jest wykres udziału utworów zawierających ścieżkę perkusyjną, co ma znaczenie z punktu widzenia charakterystyki brzmieniowej materiału.

Na podstawie przeprowadzonej analizy można stwierdzić, że zbiór danych jest niewielki liczebnie, ale zróżnicowany pod względem długości utworów i liczby nut. Oznacza to, że modele uczone na tych danych mają kontakt zarówno z krótszymi i prostszymi sekwencjami, jak i z dłuższymi, bardziej gęstymi przypadkami.

![Rozkład liczby nut w utworach.](Note_count_distribution.png)
**Rysunek 1.** Rozkład liczby nut w utworach.

![Rozkład długości ścieżek MIDI w zbiorze danych.](Track_len_dirtribution.png)
**Rysunek 2.** Rozkład długości ścieżek MIDI w zbiorze danych.  

Dodatkowo zdecydowana większość utworów zawiera ścieżkę perkusyjną, dokładnie 92,86% zbioru. Rzutowało to na ostateczną dodatkową implementacje tokenów perkusji z uwagi na jej obecność w większości utworów jak i istotność w tym stylu muzycznym.

![Udział utworów zawierających ścieżkę perkusyjną.](contain_drums.png)
**Rysunek 3.** Udział utworów zawierających ścieżkę perkusyjną.  

## 5. Opis danych

W projekcie wykorzystano zbiór plików MIDI zawierających muzykę z gier *Doom* i *Doom II*. Dane zostały pobrane z dwóch archiwów: [`doommusic.zip`](https://archive.org/download/doommusic) oraz [`doom2music.zip`](https://archive.org/download/doom2music), a po ich rozpakowaniu uzyskano łącznie 67 surowych plików MIDI.

Wybrany materiał ma charakter symboliczny, co oznacza, że każdy utwór zapisany jest jako zbiór zdarzeń muzycznych, takich jak nuty, instrumenty czy informacje czasowe, a nie jako sygnał audio. Taka forma reprezentacji jest wygodna w zadaniu generowania muzyki, ponieważ umożliwia analizę struktury utworu bez konieczności pracy na surowej fali dźwiękowej.

### 5.1 Czyszczenie danych

W ramach wstępnej analizy dla każdego pliku wyznaczono podstawowe statystyki, obejmujące między innymi czas trwania, liczbę instrumentów, liczbę nut, średnie tempo oraz informację o obecności ścieżki perkusyjnej. Na tej podstawie przeprowadzono czyszczenie zbioru: wykonano deduplikację po liczbie nut, odrzucono utwory krótsze niż 60 sekund oraz usunięto plik `dbunny.mid`, który stylistycznie odbiegał od pozostałych utworów. W rezultacie końcowy zbiór zawierał 42 utwory przeznaczone do dalszego przetwarzania.

Po oczyszczeniu danych wykonano analizę eksploracyjną zbioru. Rozkład liczby nut w utworach pozwala ocenić zróżnicowanie gęstości muzycznej pomiędzy ścieżkami, natomiast rozkład długości utworów pokazuje, że zbiór obejmuje zarówno krótsze kompozycje, jak i wyraźnie dłuższe sekwencje. Uzupełnieniem tej analizy jest wykres udziału utworów zawierających ścieżkę perkusyjną, co ma znaczenie z punktu widzenia rytmicznego charakteru materiału.

Na podstawie przeprowadzonej analizy można stwierdzić, że zbiór danych jest niewielki liczebnie, ale zróżnicowany pod względem długości utworów i liczby nut. Oznacza to, że modele uczone na tych danych mają kontakt zarówno z krótszymi i prostszymi sekwencjami, jak i z dłuższymi, bardziej gęstymi przypadkami.

![Rozkład liczby nut w utworach.](Note_count_distribution.png)
**Rysunek 1.** Rozkład liczby nut w utworach.

![Rozkład długości ścieżek MIDI w zbiorze danych.](Track_len_dirtribution.png)
**Rysunek 2.** Rozkład długości ścieżek MIDI w zbiorze danych.

Dodatkowo zdecydowana większość utworów zawiera ścieżkę perkusyjną, dokładnie 92,86% zbioru. Miało to wpływ na przyjętą reprezentację danych, ponieważ w tokenizacji uwzględniono osobny typ tokenów dla zdarzeń perkusyjnych. Jest to uzasadnione zarówno częstą obecnością perkusji w zbiorze, jak i jej istotną rolą w stylistyce muzyki z serii *Doom*.

![Udział utworów zawierających ścieżkę perkusyjną.](contain_drums.png)
**Rysunek 3.** Udział utworów zawierających ścieżkę perkusyjną.

### 5.1 Reprezentacja i tokenizacja danych

Aby umożliwić uczenie modeli sekwencyjnych, każdy utwór MIDI został przekształcony do reprezentacji event-based, inspirowanej podejściami stosowanymi w modelach generowania muzyki symbolicznej. Zamiast kodować muzykę jako sztywną siatkę pianorollową, zastosowano sekwencję tokenów opisujących konkretne zdarzenia muzyczne zachodzące w czasie. Takie podejście lepiej zachowuje strukturę utworu i pozwala reprezentować rytm, jak i instrumentację.

W implementacji przyjęto parametr `STEPS_PER_BEAT = 16`, co oznacza próbkowanie czasu do szesnastu kroków na uderzenie. Dodatkowo zastosowano `MAX_SHIFT = 64`, dzięki czemu pojedynczy token przesunięcia czasu mógł reprezentować maksymalnie 64 kroki czasowe. Pozwalało to ograniczyć długość sekwencji bez utraty podstawowej informacji rytmicznej.

Zdefiniowany słownik składał się z następujących typów tokenów:
- `PAD` – token do wyrównywania długości sekwencji,
- `BOS` – znacznik początku sekwencji,
- `EOS` – znacznik końca sekwencji,
- `NOTE_ON_x` – rozpoczęcie nuty o wysokości `x`,
- `NOTE_OFF_x` – zakończenie nuty o wysokości `x`,
- `DRUM_x` – zdarzenie perkusyjne o wysokości `x`,
- `PROGRAM_x` – zmiana instrumentu na program `x`,
- `SHIFT_x` – przesunięcie czasu o `x` kroków.

Dla każdego zdarzenia melodycznego wykorzystywano osobne tokeny `NOTE_ON` i `NOTE_OFF`, co pozwalało jawnie reprezentować czas trwania nut. Zdarzenia perkusyjne kodowano inaczej, za pomocą pojedynczego tokenu `DRUM_x`, bez osobnego `NOTE_OFF`, ponieważ perkusja ma w praktyce charakter krótkiego, jednorazowego uderzenia. Przed wystąpieniem nuty emitowany był również token `PROGRAM_x`, dzięki czemu model otrzymywał informację o aktualnym instrumencie.

Ruch w czasie nie był kodowany bezwzględnymi pozycjami, lecz za pomocą tokenów `SHIFT_x`. Oznacza to, że jeśli pomiędzy kolejnymi zdarzeniami występowała przerwa, była ona zapisywana jako odpowiednie przesunięcie czasu.

Wysokości dźwięków oraz programy instrumentów były kodowane w zakresie standardu MIDI od 0 do 127. Dla każdej możliwej wartości wysokości zdefiniowano osobne tokeny `NOTE_ON`, `NOTE_OFF` oraz `DRUM`, a dla każdego programu instrumentu osobny token `PROGRAM`. Uzupełnieniem były tokeny `SHIFT_1` do `SHIFT_64` oraz tokeny specjalne. Łączny rozmiar słownika wyniósł 579 tokenów.

W obecnej wersji projektu nie implementowano osobnych tokenów dynamiki. Prędkość nuty została ustawiona na stałą wartość `velocity = 100`, co upraszcza reprezentację, ale jednocześnie ogranicza ekspresję generowanej muzyki.

## 6. Opis metod

W projekcie problem generowania muzyki potraktowano jako zadanie autoregresyjnego modelowania sekwencji. Oznacza to, że model na podstawie wcześniej występujących tokenów przewiduje kolejny element sekwencji, a następnie proces ten jest powtarzany aż do uzyskania pełnego utworu. Wejściem dla modeli były sekwencje tokenów opisujących zdarzenia MIDI, takie jak rozpoczęcie i zakończenie nuty, zmiana instrumentu, zdarzenia perkusyjne oraz przesunięcia czasowe.

### 6.1 LSTM

Pierwszą zastosowaną metodą był model LSTM (Long Short-Term Memory), czyli odmiana rekurencyjnej sieci neuronowej zaprojektowanej do pracy z danymi sekwencyjnymi. LSTM jest rodzajem  zaawansowanej sztucznej sieci neuronowej, która potrafi zapamiętywać informacje o wcześniejszych elemenatach sekwencji. Dzięki temu jest w stanie uwzględnić kontekst historyczny podczas przewidywań. Doskonale sprawdzaja sie przy przetwarzaniu języka naturalnego, rozpoznawaniu mowy, czy generowaniu tekstu i muzyki. Podczas gdy tradycyjne sieci neuronowe mają problem z pamiętaniem zdarzeń, które miały miejsce na początku długiej sekwencji (tzw. problem zanikającego gradientu). LSTM rozwiązuje ten problem za pomocą specjalnych "bramek" w swoich komórkach pamięci: 
- **Bramka zapominająca**: decyduje, które informacje z przeszłości są zbędne i należy je odrzucić
- **Bramka wejściowa**: decyduje, które nowe informacje są ważne i zostaną dodane do pamięci
- **Bramka wyjściowa**: decyduje, jakie informacje zostaną przekazane dalej jako wynik działania sieci.

### 6.2 Model transformerowy

Drugą zastosowaną metodą był model Transformer w wariancie decoder-only. W przeciwieństwie do LSTM nie przetwarza on sekwencji wyłącznie krok po kroku poprzez stan ukryty, lecz wykorzystuje mechanizm self-attention, który pozwala analizować zależności pomiędzy elementami całego kontekstu wejściowego. Transformery umożliwiają przetwarzanie równoległe, dzięki czemu mogą być trenowane na gigantycznych zbiorach danych przy użyciu nowoczesnych procesorów graficznych. Transformery wykorzystywane są w najpopularniejszych modelach językowych, takich jak GPT, Claude czy Gemini, a także w zaawansowanych systemach generujących muzykę i obrazy.

### 6.3 Generowanie sekwencji

Po wytrenowaniu obu modeli możliwe było generowanie nowych sekwencji muzycznych. Proces generowania polegał na podaniu modelowi początkowego fragmentu sekwencji, a następnie iteracyjnym przewidywaniu kolejnych tokenów. Otrzymana sekwencja tokenów była następnie dekodowana z powrotem do formatu MIDI, co umożliwiało odsłuch i ocenę wygenerowanego materiału muzycznego.